In [1]:
from pathlib import Path
import re

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import mmread
from tqdm.notebook import tqdm

spatial_dir = Path('/compute/oven-0-13/jwinchel/spatial_adata/heimdall_inference')
split_parent_dir = Path('/home/jwinchel/Syndri/data/brca_split_output_by_sample')

layer_name = 'split_purified_counts'
has_layer_col = 'has_split_purified_counts'
write_in_place = True
output_dir = None

spatial_files = sorted(
    spatial_dir.glob('GSM*_rctd_annotated_val_med_with_admix_probs.h5ad')
)

sample_id_pattern = re.compile(r'^(GSM.+)_rctd_annotated_val_med_with_admix_probs\.h5ad$')


def sample_id_from_spatial_file(path):
    match = sample_id_pattern.match(path.name)
    if match is None:
        raise ValueError(f'Could not parse sample id from {path}')
    return match.group(1)


def read_names(path):
    return pd.read_csv(path, header=None)[0].astype(str).to_numpy()


def split_paths_for_sample(sample_id):
    sample_dir = split_parent_dir / sample_id
    return {
        'sample_dir': sample_dir,
        'matrix': sample_dir / 'split_purified_counts.mtx',
        'cells': sample_dir / 'split_purified_counts_cells.txt',
        'genes': sample_dir / 'split_purified_counts_genes.txt',
    }


def require_existing(paths):
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing SPLIT export files:\n' + '\n'.join(missing))


def build_aligned_split_layer(adata, split_matrix_path, split_cells_path, split_genes_path):
    split_cells = read_names(split_cells_path)
    split_genes = read_names(split_genes_path)
    split_counts = mmread(split_matrix_path).tocsr()

    expected_shape = (len(split_genes), len(split_cells))
    if split_counts.shape != expected_shape:
        raise ValueError(
            f'SPLIT matrix shape {split_counts.shape} does not match '
            f'{len(split_genes)} genes x {len(split_cells)} cells'
        )

    obs_positions = pd.Series(np.arange(adata.n_obs), index=adata.obs_names)
    var_positions = pd.Series(np.arange(adata.n_vars), index=adata.var_names)

    split_cell_to_obs = obs_positions.reindex(split_cells).to_numpy()
    split_gene_to_var = var_positions.reindex(split_genes).to_numpy()

    split_cell_found = ~pd.isna(split_cell_to_obs)
    split_gene_found = ~pd.isna(split_gene_to_var)

    if not split_gene_found.any():
        raise ValueError('No SPLIT genes matched adata.var_names')
    if not split_cell_found.any():
        raise ValueError('No SPLIT cells matched adata.obs_names')

    split_cell_to_obs = split_cell_to_obs.astype('float64')
    split_gene_to_var = split_gene_to_var.astype('float64')

    split_counts_cell_gene = split_counts.T.tocoo()
    valid_entries = (
        ~np.isnan(split_cell_to_obs[split_counts_cell_gene.row])
        & ~np.isnan(split_gene_to_var[split_counts_cell_gene.col])
    )

    aligned_layer = sparse.csr_matrix(
        (
            split_counts_cell_gene.data[valid_entries],
            (
                split_cell_to_obs[split_counts_cell_gene.row[valid_entries]].astype(np.int64),
                split_gene_to_var[split_counts_cell_gene.col[valid_entries]].astype(np.int64),
            ),
        ),
        shape=adata.shape,
    )

    return aligned_layer, split_cells, split_genes, split_cell_found, split_gene_found


def output_path_for_spatial_file(spatial_file):
    if write_in_place:
        return spatial_file
    if output_dir is None:
        raise ValueError('Set output_dir when write_in_place is False')
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / spatial_file.name


run_rows = []

for spatial_file in tqdm(spatial_files):
    sample_id = sample_id_from_spatial_file(spatial_file)
    split_paths = split_paths_for_sample(sample_id)
    require_existing([split_paths['matrix'], split_paths['cells'], split_paths['genes']])

    adata = ad.read_h5ad(spatial_file)
    aligned_layer, split_cells, split_genes, split_cell_found, split_gene_found = build_aligned_split_layer(
        adata,
        split_paths['matrix'],
        split_paths['cells'],
        split_paths['genes'],
    )

    adata.layers[layer_name] = aligned_layer
    adata.obs[has_layer_col] = adata.obs_names.isin(split_cells)
    adata.uns[layer_name] = {
        'source_matrix': str(split_paths['matrix']),
        'source_cells': str(split_paths['cells']),
        'source_genes': str(split_paths['genes']),
        'orientation_in_source': 'genes_x_cells',
        'orientation_in_layer': 'cells_x_genes',
        'n_split_cells': int(len(split_cells)),
        'n_split_cells_matched': int(split_cell_found.sum()),
        'n_adata_cells_without_split': int((~adata.obs[has_layer_col]).sum()),
        'n_split_genes': int(len(split_genes)),
        'n_split_genes_matched': int(split_gene_found.sum()),
    }

    write_path = output_path_for_spatial_file(spatial_file)
    adata.write_h5ad(write_path, compression='gzip')

    run_rows.append({
        'sample_id': sample_id,
        'spatial_file': str(spatial_file),
        'write_path': str(write_path),
        'n_obs': int(adata.n_obs),
        'n_vars': int(adata.n_vars),
        'n_split_cells': int(len(split_cells)),
        'n_split_cells_matched': int(split_cell_found.sum()),
        'n_adata_cells_without_split': int((~adata.obs[has_layer_col]).sum()),
        'n_split_genes': int(len(split_genes)),
        'n_split_genes_matched': int(split_gene_found.sum()),
    })

summary = pd.DataFrame(run_rows)
summary_path = split_parent_dir / 'add_split_counts_to_anndata_manifest.csv'
summary.to_csv(summary_path, index=False)
summary


  0%|          | 0/19 [00:00<?, ?it/s]

,sample_id,spatial_file,write_path,n_obs,n_vars,n_split_cells,n_split_cells_matched,n_adata_cells_without_split,n_split_genes,n_split_genes_matched
0,GSM9509156_breast_breast_B17_B17,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,68181,277,68170,68170,11,277,277
1,GSM9509157_breast_breast_B16_B16,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,15990,277,15982,15982,8,277,277
2,GSM9509158_breast_breast_B15_B15,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,33494,277,33477,33477,17,277,277
3,GSM9509159_breast_breast_B14_B14,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,77383,277,77383,77383,0,277,277
4,GSM9509160_breast_breast_B2_B2,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,130957,277,130877,130877,80,277,277
5,GSM9509161_breast_breast_B13_B13,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,37542,277,37514,37514,28,277,277
6,GSM9509162_breast_breast_B12_B12,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,151364,277,151341,151341,23,277,277
7,GSM9509163_breast_breast_B11_B11,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,79547,277,79419,79419,128,277,277
8,GSM9509164_breast_breast_B10_B10,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,66079,277,66076,66076,3,277,277
9,GSM9509165_breast_breast_B4_B4,/compute/oven-0-13/jwinchel/spatial_adata/heim...,/compute/oven-0-13/jwinchel/spatial_adata/heim...,37902,277,37900,37900,2,277,277
